In [0]:
%sql
use catalog classic_workspace;
use schema default;

In [0]:
%python
from pyspark.sql.functions import col, current_timestamp, expr

stream_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load() \
    .select(
        col("value").cast("int").alias("event_id"),
        expr("CASE WHEN value % 3 = 0 THEN 'purchase' "
             "     WHEN value % 3 = 1 THEN 'click' "
             "     ELSE 'view' END").alias("event_type"),
        expr("cast(500 + (value % 100) as int)").alias("user_id"),
        current_timestamp().alias("event_ts")
    )

In [0]:
%python
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/default/temp/checkpoints/events_stream/") \
    .trigger(processingTime="5 seconds") \
    .toTable("events")

print("Streaming query started. Writing to main.default.events every 5 seconds.")

In [0]:
%python
count_1 = spark.table("events").count()
print(f"Row count (read 1): {count_1}")

In [0]:
%python
count_2 = spark.table("events").count()
print(f"Row count (read 2): {count_2}")
print(f"New rows landed by stream: {count_2 - count_1}")

In [0]:
%python
query.stop()
print("Stream stopped.")

In [0]:
%python
final_count = spark.table("events").count()
print(f"Final row count: {final_count}")

In [0]:
describe history events;